In [68]:
from dotenv import load_dotenv
from openai import OpenAI
from PyPDF2 import PdfReader
import os
import gradio as gr
from pydantic import BaseModel
import requests

In [50]:
class Evaluation(BaseModel):
    is_acceptable : bool
    feedback : str


In [51]:
load_dotenv(override=True)
client = OpenAI(
    base_url= "https://api.groq.com/openai/v1",
    api_key=os.getenv("GROQ_API_KEY"),
)

In [52]:
gemini = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"), 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [ ]:
webhook_url = os.getenv("WEBHOOK_URL")
print(webhook_url)

None


In [75]:
def send_discord_message(message):
    data = {"content": message}
     
    response = requests.post(webhook_url, json=data)
    if response.status_code == 204:
        print("Alert sent to Discord!")
    else:
        print(f"Failed: {response.status_code}, {response.text}")


In [76]:
send_discord_message("Hello, Faisal!")

MissingSchema: Invalid URL 'None': No scheme supplied. Perhaps you meant https://None?

In [53]:
reader = PdfReader("about-me/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

print(linkedin)

   
Contact
+923054052725  (Mobile)
faisalharoon500@gmail.com
www.linkedin.com/in/faisal-
haroon500  (LinkedIn)
portfolio-1-0-rho.vercel.app/
(Personal)
Top Skills
MERN Stack
Next.js
Software DevelopmentFaisal Haroon
Learning. Creating. Evolving in AI.
Lahore, Punjab, Pakistan
Summary
Documenting my evolution in the era of AI. learning, building, and
exploring what’s next.
I started as a full-stack engineer, passionate about building things.
Over time, I realized it’s not just about tools, it’s about mindset,
curiosity, and the willingness to evolve.
Now, I’m sharpening three areas simultaneously: MERN → DevOps
→ AI. Not to chase hype, but to understand where the future is
heading.
I’m documenting everything publicly, the learning, rebuilding,
mistakes, and growth. This isn’t a polished story.
It’s a real-time journey.
Let’s see where it goes.
Experience
Self-employed
Full-Stack Engineer & AI Learner
November 2025 - Present  (6 months)
As a Full-Stack Developer and AI Learner, I dedica

In [54]:
with open("about-me/my-profile-summary.txt", "r" , encoding = "UTF-8") as f:
    summary = f.read()

print(summary)



  FAISAL HAROON — DIGITAL TWIN CONTEXT FILE
  For use as LLM system prompt / knowledge base

--- IDENTITY ---
Name: Faisal Haroon
Location: Lahore, Punjab, Pakistan
Phone: +923054052725
Email: faisalharoon500@gmail.com
LinkedIn: linkedin.com/in/faisalharoon500
Portfolio: portfolio-1-0-rho.vercel.app

--- WHO I AM ---
I'm a full-stack engineer and self-directed learner currently evolving at the intersection of MERN Stack, DevOps, and Agentic AI. I don't chase hype — I chase understanding. Right now my AI focus is specifically Agentic AI: building with LLMs, AI agents, and orchestration — not ML or data science. My journey is public, raw, and honest. I document my mistakes, my rebuilds, and my growth in real time through content creation and project building.

I believe the future belongs to engineers who pair technical skill with mindset — curiosity, adaptability, and the willingness to be a beginner again. That's the philosophy I operate from.

--- SKILLS & TECH STACK ---
Primary: MERN

In [55]:
name = "Faisal Haroon"

In [56]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n##LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."

In [57]:
messages = [{"role": "system", "content": system_prompt}] + [{"role":"user", "content":"Do you hold a patent?"}]
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=messages
)
reply = response.choices[0].message.content
print(reply)

I don't have any patents to my name. As a full-stack engineer and AI learner, my focus has been on building and learning, rather than intellectual property protection. I'm more about sharing my knowledge and experiences through content creation and public projects, rather than holding onto proprietary information. My goal is to contribute to the tech community and help others grow, rather than pursuing patents or proprietary rights.


In [58]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [59]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [60]:
def evaluate(reply, message, history) -> Evaluation:

    messages= [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = gemini.chat.completions.parse(
        model="gemini-3-flash-preview",
        messages=messages,
        response_format=Evaluation
    )
    evaluation_answer = response.choices[0].message.parsed
    return evaluation_answer

In [61]:
evaluate(reply, "Do you hold a patent?", messages[:1])

Evaluation(is_acceptable=True, feedback="The response is accurate based on the provided context. Faisal's summary and LinkedIn profile do not mention any patents, so stating that he doesn't have any is appropriate. The tone is professional and aligns well with the personal philosophy described in his profile regarding transparency and public learning.")

In [62]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = client.chat.completions.create(model="llama-3.3-70b-versatile", messages=messages)
    print(response.choices[0].message.content)
    return response.choices[0].message.content

In [63]:
def chat(message, history):
     history = [{"role": h["role"], "content": h["content"]} for h in history]
     system = system_prompt
     messages= [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
     response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=messages
        )
     reply = response.choices[0].message.content

     evaluation = evaluate(reply , message, history)
    
     if evaluation.is_acceptable:
      print("Accepted")
     else:
      print("Rejected")
      print(evaluation.feedback)
      reply = rerun(reply, message, history, evaluation.feedback)
     return reply


In [64]:
gr.ChatInterface(chat).launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


Accepted
Accepted
Accepted
